In [1]:
import os
import glob
import shutil
import subprocess

In [57]:
input_prefix = ""
input_pdb = "input/pdb"


output_dir = f"output/{input_prefix}"


#### Prepare PDB file in origin

In [58]:
# Use ananas to generate PDB model in origin
ananas_script = "helper_scripts/ananas"
ananas_out = f"{output_dir}/sym_diff/ananas"
input_basename = os.path.basename(input_pdb).split(".")[0]
os.makedirs(ananas_out, exist_ok=True)

ananas_cmd = f"{ananas_script} {input_pdb} -j {ananas_out}/sym.json --origin {ananas_out}/{input_basename}_sym.pdb"
print(ananas_cmd)


# Use ananas origin model:
input_pdb = f"{ananas_out}/{input_basename}_sym.pdb"


helper_scripts/ananas input/2B9_core.pdb -j output/2B9_core_sym/sym_diff/ananas/sym.json --origin output/2B9_core_sym/sym_diff/ananas/2B9_core_sym.pdb


In [59]:
# subprocess.run(ananas_cmd, shell=True, check=True)

In [60]:
# new pdb
input_pdb = "input/"

#### Prepare bash script for RFdiffusion and AF2 validation

Make sure to target only one assymetric unit! Example:
```bash
python /home/tsatler/RFdif/RFdiffusion/scripts/run_inference.py \
        --config-name=symmetry \
        diffuser.T=$diffuser_T \
        inference.input_pdb=$input_pdb \
        inference.num_designs=$num_designs \
        inference.symmetry="C2" \
        inference.ckpt_override_path=/home/tsatler/RFdif/RFdiffusion/models/Complex_beta_ckpt.pt \
        "inference.output_prefix=$testing_output_dir" \
        'contigmap.contigs=[A65-87/A131-198 80-80/0 A65-87/A131-198 80-80/0]' \
        ppi.hotspot_res=$hotspots \
        potentials.olig_intra_all=True \
        potentials.olig_inter_all=True \
        potentials.guide_scale=2 \
        potentials.guide_decay="quadratic"
```

In [61]:
# Slurm parameters
num_jobs = 200  # Number of jobs to submit
array_limit = 1

# RF diffusion parameters
hotspots = "[A147,A150,A154,A195,A196,A198,A199,A201,A202,A204,A205,A208]"
diffuser_T = 20
num_of_diffusions = 10  # Number of RF diffusions per job
symm = "C5"
copies = 5 # *2 if D symmetry
contigs = "A140-210 100-100/0 A140-210 100-100/0 A140-210 100-100/0 A140-210 100-100/0 A140-210 100-100/0"
# contigs = "A140-210 100-100/0 B140-210 100-100/0 C140-210 100-100/0 D140-210 100-100/0 E140-210 100-100/0"
# contigs = "A141-209/0 100-100/0 A141-209/0 100-100/0 A141-209/0 100-100/0 A141-209/0 100-100/0 A141-209/0 100-100/0"
contigs_list = contigs.split(" ")
contigs_str = ":".join(contigs_list)
# guiding_potentials = 'potentials.guiding_potentials=["type:olig_contacts,weight_intra:0.1,weight_inter:0.2"]'
guiding_potentials = 'potentials.guiding_potentials=["type:olig_contacts,weight_intra:1,weight_inter:0.1"]'
# guide_scale = 1
# guiding_potentials = "" # might be better without?
guide_scale = 2

# MPNN parameters
num_seqs = 32  # How many MPNN sequences to generate per RF diffusion
mpnn_batch = 16

# Af2 Mpnn parameters
num_recycles = 3  # AF2 recycles
sampling_temp = 0.0001  # ProteinMPNN sampling temperature

print(
    f"🔄 Submitting {num_jobs} jobs to the cluster, each generating {num_of_diffusions} RF diffusions..."
)
print(f"📊 In total, {num_jobs * num_of_diffusions} RF diffusions will be generated.")
print(
    f"🔬 For each diffusion, {num_seqs} MPNN sequences will be produced, resulting in {num_seqs * num_jobs * num_of_diffusions} sequences validated with AF2."
)

🔄 Submitting 200 jobs to the cluster, each generating 10 RF diffusions...
📊 In total, 2000 RF diffusions will be generated.
🔬 For each diffusion, 32 MPNN sequences will be produced, resulting in 64000 sequences validated with AF2.


In [62]:
os.makedirs(output_dir, exist_ok=True)
script = f"""#!/bin/bash
#SBATCH --partition=gpu
#SBATCH --gres=gpu:A40:1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=2
#SBATCH --array=0-{num_jobs-1}%{array_limit}
#SBATCH --job-name=sym_dif_{input_prefix}

set -e

source /home/tsatler/anaconda3/etc/profile.d/conda.sh
conda activate SE3nv


### RF DIFFUSION ###
rfdiff_output_dir={output_dir}/sym_diff/rf/rf_$SLURM_ARRAY_TASK_ID
rf_diff_prefix={input_prefix}_$SLURM_ARRAY_TASK_ID

mkdir -p $rfdiff_output_dir

python /home/tsatler/RFdif/RFdiffusion/scripts/run_inference.py \
        --config-name=symmetry \
        diffuser.T={diffuser_T} \
        inference.input_pdb={input_pdb} \
        inference.num_designs={num_of_diffusions} \
        inference.symmetry={symm} \
        inference.ckpt_override_path=/home/tsatler/RFdif/RFdiffusion/models/Complex_beta_ckpt.pt \
        "inference.output_prefix=$rfdiff_output_dir/$rf_diff_prefix" \
        'contigmap.contigs=[{contigs}]' \
        ppi.hotspot_res={hotspots} \
        '{guiding_potentials}' \
        potentials.olig_intra_all=True \
        potentials.olig_inter_all=True \
        potentials.guide_scale={guide_scale} \
        potentials.guide_decay="quadratic" \

### AF2 MPNN ###
echo "Running AF2 MPNN..."
conda activate colabthread

script=helper_scripts/sym_colabdesign_fix.py
input_pdbs=($rfdiff_output_dir/$rf_diff_prefix*.pdb)
af_output_dir={output_dir}/sym_diff/af2/af2_$SLURM_ARRAY_TASK_ID
mkdir -p $af_output_dir

for ((i=0; i<${{#input_pdbs[@]}}; i++)); do
        pdb=${{input_pdbs[$i]}}
        echo "Processing $pdb..."
        
        python $script $pdb $af_output_dir {contigs_str} \
                --num_seqs {num_seqs} \
                --sampling_temp {sampling_temp} \
                --num_recycles {num_recycles} \
                --rm_aa="C" \
                --copies {copies} \
                --mpnn_batch {mpnn_batch} \
                --results_dataframe {output_dir}/sym_diff \
                --initial_guess --use_multimer --use_soluble \
                --save_best_only 
done
"""
script_dir = f"{output_dir}/sym_diff"
os.makedirs(script_dir, exist_ok=True)
with open(f"{script_dir}/sym_diff_{input_prefix}.sh", "w") as f:
    f.write(script)

In [63]:
### Submit jobs
print(f"🚀 Submitting jobs...")
subprocess.run(
    f"sbatch {script_dir}/sym_diff_{input_prefix}.sh", shell=True, check=True
)

🚀 Submitting jobs...
Submitted batch job 589079


CompletedProcess(args='sbatch output/2B9_core_sym/sym_diff/sym_diff_2B9_core_sym.sh', returncode=0)